<a href="https://colab.research.google.com/github/LaurenMitchell-tech/uvvisml/blob/main/CIE_Training_With_HPO_Draft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Install chemprop from GitHub if running in Google Colab
import os

if os.getenv("COLAB_RELEASE_TAG"):
    try:
        import chemprop
    except ImportError:
        !git clone https://github.com/chemprop/chemprop.git
        %cd chemprop
        !pip install -e .
        !pip install "ray[tune]==2.50.0"
        !pip install ".[hpopt]"
        %cd examples

In [1]:
%cd chemprop
%cd examples

/content/chemprop
/content/chemprop/examples


In [5]:
from lightning import pytorch as pl
import torch
import numpy as np
import pandas as pd
from pathlib import Path

import pickle
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

#Tuning imports
import ray
from ray import tune
from ray.train import CheckpointConfig, RunConfig, ScalingConfig
from ray.train.lightning import (RayDDPStrategy, RayLightningEnvironment,
                                 RayTrainReportCallback, prepare_trainer)
from ray.train.torch import TorchTrainer
from ray.tune.search.hyperopt import HyperOptSearch
from ray.tune.search.optuna import OptunaSearch
from ray.tune.schedulers import FIFOScheduler
from ray.tune.tuner import Tuner, TuneConfig

from chemprop import data, featurizers, models, nn
from chemprop.nn import metrics
from chemprop.models import multi

In [2]:
from google.colab import files
uploaded = files.upload()


Saving Deep4Chem_CIE_Coordinates_FWHM(cm-1).csv to Deep4Chem_CIE_Coordinates_FWHM(cm-1) (1).csv


In [6]:
chemprop_dir = Path.cwd().parent
input_path = chemprop_dir / "examples" / "Deep4Chem_CIE_Coordinates_FWHM(cm-1).csv"
num_workers = 0 # number of workers for dataloader. 0 means using main process for data loading
smiles_columns = ['Chromophore', 'Solvent']
target_columns = ["X","Y"]

hpopt_save_dir = Path.cwd() / "hyperopt" # directory to save hyperopt results
hpopt_save_dir.mkdir(exist_ok=True)

In [7]:
df_input = pd.read_csv(input_path, usecols=['Chromophore','Solvent', 'X', 'Y'])
smiss = df_input.loc[:, smiles_columns].values
ys = df_input.loc[:, target_columns].values
df_input

,Chromophore,Solvent,X,Y
0,CCC(=O)c1ccc2cc(N(C)C)ccc2c1,C1CCCCC1,0.169691,0.006685
1,CCC(=O)c1ccc2cc(N(C)C)ccc2c1,c1ccccc1,0.164626,0.011095
2,CCC(=O)c1ccc2cc(N(C)C)ccc2c1,CCN(CC)CC,0.167469,0.008560
3,CCC(=O)c1ccc2cc(N(C)C)ccc2c1,Clc1ccccc1,0.157068,0.021501
4,CCC(=O)c1ccc2cc(N(C)C)ccc2c1,CC(C)=O,0.141891,0.062884
...,...,...,...,...
621,CN(C)c1ccc(C2=Cc3cccc[n+]3[B-](F)(F)O2)cc1,ClC(Cl)Cl,0.131381,0.311698
622,CN1CCc2cc(C3=Cc4cccc[n+]4[B-](F)(F)O3)ccc21,ClC(Cl)Cl,0.200724,0.546418
623,F[B-]1(F)OC(c2ccc(N3CCCC3)cc2)=Cc2cccc[n+]21,ClC(Cl)Cl,0.155260,0.369463
624,CN1CCCc2cc(C3=Cc4cccc[n+]4[B-](F)(F)O3)ccc21,ClC(Cl)Cl,0.189495,0.471387


In [8]:
datapoints = [[data.MoleculeDatapoint.from_smi(smis[0], y) for smis, y in zip(smiss, ys)]]
datapoints += [[data.MoleculeDatapoint.from_smi(smis[i]) for smis in smiss] for i in range(1, len(smiles_columns))]

In [9]:
#split by indices
component_to_split_by = 0 # index of the component to use for structure based splits
mols = [d.mol for d in datapoints[component_to_split_by]]

train_indices, val_indices, test_indices = data.make_split_indices(mols, "random", (0.8, 0.1, 0.1))
train_data, val_data, test_data = data.split_data_by_indices(
    datapoints, train_indices, val_indices, test_indices
)

In [10]:
#Display test set
test_indices = [idx for sublist in test_indices for idx in sublist]
df_test = df_input.iloc[test_indices].reset_index(drop=True)
df_test

,Chromophore,Solvent,X,Y
0,c1csc(-c2cnc(-c3cccs3)c3nonc23)c1,CO,0.539653,0.451864
1,CN(C)c1ccc(C2=Cc3c4ccccc4c4ccccc4[n+]3[B-](F)(...,C1CCOC1,0.314741,0.607312
2,c1csc(-c2cnc(-c3cccs3)c3nonc23)c1,C1CCCCC1,0.429584,0.532635
3,CC(=O)Nc1ccc(S(=O)(=O)O)c(/N=N/c2c(O)c3ccccc3n...,CCOC(C)=O,0.264535,0.521251
4,COc1cc2c(cc1/C=C/C1=CC(=C(C#N)C#N)CC(C)(C)C1)c...,ClCCl,0.592879,0.405893
...,...,...,...,...
59,CN(C)c1ccc(/C=C/c2nc3c4ccccc4c4ccccc4c3[nH]2)cc1,C1CCOC1,0.146787,0.166225
60,COc1cc2c(cc1/C=C(\C#N)c1nc3ccccc3s1)c1ccccc1n2C,C1CCOC1,0.146627,0.406056
61,O=C1CCCc2c1[nH]c1cc3c(cc21)OCO3,CCO,0.147754,0.053626
62,CC(=O)c1nc(-c2ccc(N(c3ccccc3)c3ccccc3)cc2)oc1C,CCO,0.152907,0.198400


In [11]:
#fearurizer, datasets, and dataloaders
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()

train_dset = [data.MoleculeDataset(train_data[0][i], featurizer) for i in range(len(smiles_columns))]
val_dset = [data.MoleculeDataset(val_data[0][i], featurizer) for i in range(len(smiles_columns))]
test_dset = [data.MoleculeDataset(test_data[0][i], featurizer) for i in range(len(smiles_columns))]

train_mcdset = data.MulticomponentDataset(train_dset)
scaler = train_mcdset.normalize_targets()
val_mcdset = data.MulticomponentDataset(val_dset)
val_mcdset.normalize_targets(scaler)
test_mcdset = data.MulticomponentDataset(test_dset)

In [12]:
#Function from hyperparameter tuning doc

def train_model(config, train_mcdset, val_mcdset, num_workers, scaler, smiles_columns, target_columns):

    # config is a dictionary containing hyperparameters used for the trial
    depth = int(config["depth"])
    ffn_hidden_dim = int(config["ffn_hidden_dim"])
    ffn_num_layers = int(config["ffn_num_layers"])
    message_hidden_dim = int(config["message_hidden_dim"]) #used in ffn

    #train_loader = data.build_dataloader(train_dset, num_workers=num_workers, shuffle=True)
    #val_loader = data.build_dataloader(val_dset, num_workers=num_workers, shuffle=False)

    BATCH_SIZE = 16
    train_loader = data.build_dataloader(train_mcdset, num_workers=num_workers, batch_size=BATCH_SIZE)
    val_loader = data.build_dataloader(val_mcdset, num_workers=num_workers, batch_size=BATCH_SIZE, shuffle=False)
    #test_loader = data.build_dataloader(test_mcdset, batch_size=BATCH_SIZE, shuffle=False)

    blocks = [nn.BondMessagePassing(d_h=message_hidden_dim, depth=depth) for _ in range(len(smiles_columns))]
    mcmp = nn.MulticomponentMessagePassing(blocks=blocks, n_components=len(smiles_columns),) #changed
    agg = nn.MeanAggregation() #same
    output_transform = nn.UnscaleTransform.from_standard_scaler(scaler) #same
    ffn = nn.RegressionFFN(output_transform=output_transform, input_dim=mcmp.output_dim, hidden_dim=ffn_hidden_dim, n_layers=ffn_num_layers, n_tasks = len(target_columns),)
    batch_norm = True #new
    metric_list = [nn.metrics.RMSE(), nn.metrics.MAE()] #same
    model = multi.MulticomponentMPNN(mcmp, agg, ffn, batch_norm, metric_list) #changed

    trainer = pl.Trainer( #add logger and deterministic?
        accelerator="auto",
        devices=1,
        max_epochs=20,
        # below are needed for Ray and Lightning integration
        strategy=RayDDPStrategy(),
        callbacks=[RayTrainReportCallback()],
        plugins=[RayLightningEnvironment()],
    )

    #trainer = pl.Trainer(accelerator="auto", logger=True, callbacks=[checkpoint, early_stop], max_epochs=500, deterministic=True)

    trainer = prepare_trainer(trainer)
    trainer.fit(model, train_loader, val_loader)

In [13]:
search_space = {
    "depth": tune.qrandint(lower=2, upper=6, q=1),
    "ffn_hidden_dim": tune.qrandint(lower=300, upper=2400, q=100),
    "ffn_num_layers": tune.qrandint(lower=1, upper=3, q=1),
    "message_hidden_dim": tune.qrandint(lower=300, upper=2400, q=100),
}

In [14]:
ray.init()

scheduler = FIFOScheduler()

# Scaling config controls the resources used by Ray
scaling_config = ScalingConfig(
    num_workers=1,
    use_gpu=False, # change to True if you want to use GPU
)

# Checkpoint config controls the checkpointing behavior of Ray
checkpoint_config = CheckpointConfig(
    num_to_keep=1, # number of checkpoints to keep
    checkpoint_score_attribute="val_loss", # Save the checkpoint based on this metric
    checkpoint_score_order="min", # Save the checkpoint with the lowest metric value
)

run_config = RunConfig(
    checkpoint_config=checkpoint_config,
    storage_path=hpopt_save_dir / "ray_results", # directory to save the results
)

# Directly define the TorchTrainer instance
ray_trainer = TorchTrainer(
    lambda config: train_model(
        config, train_dset, val_dset, num_workers, scaler, smiles_columns, target_columns
    ),
    scaling_config=scaling_config,
    run_config=run_config,
)

search_alg = HyperOptSearch(
    n_initial_points=1, # number of random evaluations before tree parzen estimators
    random_state_seed=42,
)

# OptunaSearch is another search algorithm that can be used
# search_alg = OptunaSearch()

tune_config = tune.TuneConfig(
    metric="val_loss",
    mode="min",
    num_samples=2, # number of trials to run
    scheduler=scheduler,
    search_alg=search_alg,
    trial_dirname_creator=lambda trial: str(trial.trial_id), # shorten filepaths

)

tuner = tune.Tuner(
    ray_trainer, # Pass the TorchTrainer instance directly
    param_space={
        "train_loop_config": search_space,
    },
    tune_config=tune_config,
)

# Start the hyperparameter search
results = tuner.fit()

2025-12-29 23:59:24,541	INFO worker.py:2013 -- Started a local Ray instance.
/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


+---------------------------------------------------------------------+
| Configuration for experiment     TorchTrainer_2025-12-29_23-59-36   |
+---------------------------------------------------------------------+
| Search algorithm                 SearchGenerator                    |
| Scheduler                        FIFOScheduler                      |
| Number of trials                 2                                  |
+---------------------------------------------------------------------+

View detailed results here: /content/chemprop/examples/hyperopt/ray_results/TorchTrainer_2025-12-29_23-59-36
To visualize your results with TensorBoard, run: `tensorboard --logdir /tmp/ray/session_2025-12-29_23-59-18_434096_2700/artifacts/2025-12-29_23-59-36/TorchTrainer_2025-12-29_23-59-36/driver_artifacts`

Trial status: 1 PENDING
Current time: 2025-12-29 23:59:38. Total running time: 0s
Logical resource usage: 0/2 CPUs, 0/0 GPUs
+----------------------------------------------------------

(TorchTrainer pid=3480) Started distributed worker processes: 
(TorchTrainer pid=3480) - (node_id=bf98860beaa939da21c0cbf2e5e1241036ec06defb0f46f62a68c0e9, ip=172.28.0.12, pid=3594) world_rank=0, local_rank=0, node_rank=0
(RayTrainWorker pid=3594) Setting up process group for: env:// [rank=0, world_size=1]


(RayTrainWorker pid=3594) [Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0

Trial TorchTrainer_d9e05476 started with configuration:
+---------------------------------------------+
| Trial TorchTrainer_d9e05476 config          |
+---------------------------------------------+
| train_loop_config/depth                   2 |
| train_loop_config/ffn_hidden_dim       2200 |
| train_loop_config/ffn_num_layers          2 |
| train_loop_config/message_hidden_dim    400 |
+---------------------------------------------+


(TrainTrainable pid=3593) Trainable.setup took 12.827 seconds. If your trainable is slow to initialize, consider setting reuse_actors=True to reduce actor creation overheads.
(RayTrainWorker pid=3594) 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
(RayTrainWorker pid=3594) GPU available: False, used: False
(RayTrainWorker pid=3594) TPU available: False, using: 0 TPU cores
(RayTrainWorker pid=3594) 2025-12-30 00:00:25.999559: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
(RayTrainWorker pid=3594) 2025-12-30 00:00:26.015237: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
(RayTrainWorker pid=3594) 2025-12-30 00:00:26.072024: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Un

...

(RayTrainWorker pid=3594) ┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
(RayTrainWorker pid=3594) ┃   ┃ Name            ┃ Type                         ┃ Params ┃ Mode  ┃ FLOPs ┃
(RayTrainWorker pid=3594) ┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
(RayTrainWorker pid=3594) │ 0 │ message_passing │ MulticomponentMessagePassing │  1.2 M │ train │     0 │
(RayTrainWorker pid=3594) │ 1 │ agg             │ MeanAggregation              │      0 │ train │     0 │
(RayTrainWorker pid=3594) │ 2 │ bn              │ BatchNorm1d                  │  2.0 K │ train │     0 │
(RayTrainWorker pid=3594) │ 3 │ predictor       │ RegressionFFN                │  6.0 M │ train │     0 │
(RayTrainWorker pid=3594) │ 4 │ X_d_transform   │ Identity                     │      0 │ train │     0 │
(RayTrainWorker pid=3594) │ 5 │ metrics         │ ModuleList                   │      0 │ train │     0 │
(RayTrainWorker pid=3594) └───┴───────────────

(RayTrainWorker pid=3798) 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
(RayTrainWorker pid=3798) GPU available: False, used: False
(RayTrainWorker pid=3798) TPU available: False, using: 0 TPU cores
(RayTrainWorker pid=3798) 2025-12-30 00:00:46.804898: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
(RayTrainWorker pid=3798) 2025-12-30 00:00:46.810667: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
(RayTrainWorker pid=3798) 2025-12-30 00:00:46.829598: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
(RayTrainWorker pid=3798) WARNING: All log messages before

(RayTrainWorker pid=3798) │ 0 │ message_passing │ MulticomponentMessagePassing │  767 K │ train │     0 │
(RayTrainWorker pid=3798) ┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
(RayTrainWorker pid=3798) ┃   ┃ Name            ┃ Type                         ┃ Params ┃ Mode  ┃ FLOPs ┃
(RayTrainWorker pid=3798) ┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
(RayTrainWorker pid=3798) │ 1 │ agg             │ MeanAggregation              │      0 │ train │     0 │
(RayTrainWorker pid=3798) │ 2 │ bn              │ BatchNorm1d                  │  1.6 K │ train │     0 │
(RayTrainWorker pid=3798) │ 3 │ predictor       │ RegressionFFN                │  6.6 M │ train │     0 │
(RayTrainWorker pid=3798) │ 4 │ X_d_transform   │ Identity                     │      0 │ train │     0 │
(RayTrainWorker pid=3798) │ 5 │ metrics         │ ModuleList                   │      0 │ train │     0 │
(RayTrainWorker pid=3798) └───┴───────────────

2025-12-30 00:00:52,076	ERROR tune_controller.py:1331 -- Trial task failed for trial TorchTrainer_d9e05476
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/ray/air/execution/_internal/event_manager.py", line 110, in resolve_future
    result = ray.get(future)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ray/_private/auto_init_hook.py", line 22, in auto_init_wrapper
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ray/_private/client_mode_hook.py", line 104, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py", line 2962, in get
    values, debugger_breakpoint = worker.get_objects(
                                  ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py", line 1026, in get_objects
    raise value.as_instanceof_cause(


Trial TorchTrainer_d9e05476 errored after 0 iterations at 2025-12-30 00:00:52. Total running time: 1min 13s
Error file: /tmp/ray/session_2025-12-29_23-59-18_434096_2700/artifacts/2025-12-29_23-59-36/TorchTrainer_2025-12-29_23-59-36/driver_artifacts/d9e05476/error.txt

Trial status: 2 ERROR
Current time: 2025-12-30 00:00:52. Total running time: 1min 13s
Logical resource usage: 1.0/2 CPUs, 0/0 GPUs
+--------------------------------------------------------------------------------------------------------------------------------------+
| Trial name              status       ...loop_config/depth     ...ig/ffn_hidden_dim     ...ig/ffn_num_layers     ...essage_hidden_dim |
+--------------------------------------------------------------------------------------------------------------------------------------+
| TorchTrainer_437daaf4   ERROR                           2                     2000                        2                      500 |
| TorchTrainer_d9e05476   ERROR                    

In [13]:
#This cell is broken, probably delete.
ray.init()

scheduler = FIFOScheduler()

# Scaling config controls the resources used by Ray
scaling_config = ScalingConfig(
    num_workers=1,
    use_gpu=True, # change to True if you want to use GPU
)

# Checkpoint config controls the checkpointing behavior of Ray
checkpoint_config = CheckpointConfig(
    num_to_keep=1, # number of checkpoints to keep
    checkpoint_score_attribute="val_loss", # Save the checkpoint based on this metric
    checkpoint_score_order="min", # Save the checkpoint with the lowest metric value
)

run_config = RunConfig(
    checkpoint_config=checkpoint_config,
    storage_path=hpopt_save_dir / "ray_results", # directory to save the results
    #name="chemprop_tune_experiment" # Explicitly name the experiment-Gemini
)

'''
ray_trainer = TorchTrainer(
    lambda config: train_model(
        config, train_dset, val_dset, num_workers, scaler
    ),
    scaling_config=scaling_config,
    run_config=run_config,
)
'''

search_alg = HyperOptSearch(
    n_initial_points=1, # number of random evaluations before tree parzen estimators
    random_state_seed=42,
)

# OptunaSearch is another search algorithm that can be used
# search_alg = OptunaSearch()

# TorchTrainer wraps your train_model function
ray_trainer = TorchTrainer(
    train_loop_per_worker=lambda config: train_model(
        config=config,
        train_dset=train_dset,
        val_dset=val_dset,
        num_workers=num_workers,
        scaler=scaler,
        smiles_columns=smiles_columns,
        target_columns=target_columns,
    ),
    train_loop_config=search_space,  # hyperparameter search space
    scaling_config=scaling_config,
    run_config=run_config,
)

# Tuner config
tune_config = tune.TuneConfig(
    metric="val_loss",
    mode="min",
    num_samples=2, # number of trials to run
    scheduler=scheduler,
    search_alg=search_alg,
)

# Initialize the Tuner
tuner = tune.Tuner(
    ray_trainer,
    tune_config=tune_config
)

# Run the hyperparameter search
results = tuner.fit()

'''
tune_config = tune.TuneConfig(
    metric="val_loss",
    mode="min",
    num_samples=2, # number of trials to run
    scheduler=scheduler,
    search_alg=search_alg,
    trial_dirname_creator=lambda trial: str(trial.trial_id), # shorten filepaths

)

tuner = tune.Tuner(
    ray_trainer,
    param_space={
        "train_loop_config": search_space,
    },
    tune_config=tune_config,
    #run_config=run_config # Pass the run_config to tune.Tuner for overall experiment management-Gemini
)

# Start the hyperparameter search
results = tuner.fit()
'''

RuntimeError: Maybe you called ray.init twice by accident? This error can be suppressed by passing in 'ignore_reinit_error=True' or by calling 'ray.shutdown()' prior to 'ray.init()'.

In [ ]:
results

In [ ]:
# results of all trials
result_df = results.get_dataframe()
result_df

In [ ]:
# best configuration
best_result = results.get_best_result()
best_config = best_result.config
best_config['train_loop_config']

In [ ]:
# best model checkpoint path
best_result = results.get_best_result()
best_checkpoint_path = Path(best_result.checkpoint.path) / "checkpoint.ckpt"
print(f"Best model checkpoint path: {best_checkpoint_path}")

In [25]:
ray.shutdown()

From here down is the old multicomponent code, most of which will be wrapped into the train_model function above

In [ ]:
#multicomponent code

torch.manual_seed(0) #use same random seed every time
np.random.seed(0)

mcmp = nn.MulticomponentMessagePassing(
    blocks=[nn.BondMessagePassing() for _ in range(len(smiles_columns))],
    n_components=len(smiles_columns),
)

agg = nn.MeanAggregation()

In [ ]:
#multicomponent code, could add dropout in the FFN (e.g., dropout=0.2–0.5)
output_transform = nn.UnscaleTransform.from_standard_scaler(scaler)

ffn = nn.RegressionFFN(
    n_tasks = len(target_columns),
    input_dim=mcmp.output_dim,
    output_transform=output_transform,
)

metric_list = [metrics.RMSE(), metrics.MAE()] # Only the first metric is used for training and early stopping

In [ ]:
#multicomponent code
mcmpnn = multi.MulticomponentMPNN(
    mcmp,
    agg,
    ffn,
    metrics=metric_list,
)

mcmpnn

In [ ]:
#code for training a model which will be saved
checkpoint = pl.callbacks.ModelCheckpoint(
    dirpath="checkpoints/",
    monitor="val_loss",
    save_top_k=1,
    mode="min",
    filename="CIE_model_1"
)

early_stop = pl.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=50,     # stop if val_loss hasn’t improved for x epochs
    mode="min"
)

from torch.optim.lr_scheduler import ReduceLROnPlateau

optimizer = torch.optim.Adam(mcmpnn.parameters(), lr=1e-4, weight_decay=1e-5) #Use mcmpnn or chemprop_model followed by .parameters()
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10) #patience adjusts lr if no improvment after x epochs

#Recommended lr = 1e-4 or 5e-4 for small dataset (< 10,000 datapoints)
#chemprop.args.TrainArgs(learning_rate=0.0005) #or I can use this built-in, simpler line

In [ ]:
#multicomponent code
trainer = pl.Trainer(accelerator="auto", logger=True, callbacks=[checkpoint, early_stop], max_epochs=500, deterministic=True)

In [ ]:
#multicomponent code
trainer.fit(mcmpnn, train_loader, val_loader)

In [ ]:
#Predict and show results compared to known data
preds = trainer.predict(mcmpnn, test_loader) #Works with mcmpnn or chemprop_model
preds_tensor = torch.cat(preds, dim=0)

preds_array = preds_tensor.detach().cpu().numpy()

columns = ['Pred_X', 'Pred_Y']
df_test[columns] = preds_array

df_test

In [ ]:
#Show where model is saved
best_model_path = checkpoint.best_model_path
print(f"Best model saved at: {best_model_path}")

Make parity plots of model predictions

In [ ]:
def plot_and_save(true_values=str, preds=str, model=str, value_type=str, save_path=str):

  '''
  true_values: name of column containing experimental x or y values
  preds: name of column containing predicted x or y values
  model: name of model used to make prediction
  save_path: path where figure will be saved

  plots a parity plot and saves the figure to the specified path
  '''

  true_ys = df_test[true_values].tolist()
  preds_y = df_test[preds].tolist()

  fig = plt.figure(figsize=(8,5))
  plt.scatter(true_ys, preds_y, color='red', marker='o', label='Parity Plot')
  ax = plt.gca()
  lims = ax.get_xlim()
  ax.plot(lims, lims, color='black')
  plt.legend()
  plt.xlabel('Actual')
  plt.ylabel('Predicted')
  title = 'Parity Plot of ' + model + ' ' + value_type + ' Values'
  plt.title(title)
  plt.grid(True)

  with open(save_path, "wb") as f:
      pickle.dump(fig, f)

  plt.show()

  print('Plot saved as: ' + save_path)

In [ ]:
plot_and_save('X', 'Pred_X', 'CIE_model_1', 'X', 'X_Parity_CIE_model_1.pkl')

In [ ]:
plot_and_save('Y', 'Pred_Y', 'CIE_model_1', 'Y', 'Y_Parity_CIE_model_1.pkl')